In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Imports

In [2]:
! pip install -q chonkie sentence-transformers faiss-cpu jsonlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.8/233.8 kB 6.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 63.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.2/387.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 55.3 MB/s eta 0:00:00:00:01


In [31]:
from kaggle_secrets import UserSecretsClient
import wandb

from chonkie import TokenChunker

import numpy as np              
import pandas as pd             
import matplotlib.pyplot as plt 
import seaborn as sns           
import torch                    
import torch.nn as nn           
from collections import Counter 
from string import punctuation  
import warnings                 
import string
import re

from transformers import pipeline,AutoTokenizer,AutoModel,AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, CrossEncoder 
import faiss 

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

from tqdm import tqdm
import uuid
import jsonlines

from datasets import load_dataset

In [4]:
%matplotlib inline
plt.style.use('fivethirtyeight')
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

print(f"PyTorch Version: {torch.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
np.random.seed(42)

PyTorch Version: 2.10.0+cpu
NumPy Version: 2.4.6
Pandas Version: 2.3.3
CUDA Available: False


# W&B

In [5]:
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb_api")

In [6]:
wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [7]:
CONFIG={
    "project_name":"23f2000391-t22026",
    "model":"RAG",
    "knowledge_base":"Wikipedia",
    "embedding_model":"BAAI/bge-base-en-v1.5",
    "retriever":"FAISS IndexFlatIP",
    "cross-encoder":"cross-encoder/ms-marco-MiniLM-L-6-v2",
    "llm":"Qwen2.5-7B-Instruct",
    "chunk_startegy":"Token Chunking",
    "chunk_size":256,
    "chunk_overlap":32,
    "retrieve_k":5,
    "split":"Stratified-K-Fold",
    "folds_num":5
}

In [8]:
wandb.init(
    project=CONFIG["project_name"],
    name='RAG_wikipedia',
    config=CONFIG
)

wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


# Dataset

In [9]:
train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [10]:
train.drop(columns='id',inplace=True)
train.head()

,prompt,A,B,C,D,E,answer
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [11]:
test=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
test.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [12]:
test.drop(columns='id',inplace=True)
test.head()

,prompt,A,B,C,D,E
0,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


# EDA

In [9]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   prompt  2000 non-null   object
 1   A       2000 non-null   object
 2   B       2000 non-null   object
 3   C       2000 non-null   object
 4   D       2000 non-null   object
 5   E       2000 non-null   object
 6   answer  2000 non-null   object
dtypes: object(7)
memory usage: 109.5+ KB


In [10]:
train.describe()

,prompt,A,B,C,D,E,answer
count,2000,2000,2000,2000,2000,2000,2000
unique,1758,316,328,303,318,320,5
top,Select the most accurate option: How do the Lu...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,B
freq,4,21,21,21,21,21,490


There might be some duplicates since only 1758 out of the 2000 prompts are unique

## Duplicates

In [13]:
print(f"Number of Duplicates: {train.duplicated().sum()}")

Number of Duplicates: 183


In [14]:
print("Shape before dropping duplicates",train.shape)
train.drop_duplicates(inplace=True)
print("Shape after dropping duplicates",train.shape)

Shape before dropping duplicates (2000, 7)
Shape after dropping duplicates (1817, 7)


## Class Distribution

In [12]:
train['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

## Prompt Word Length

In [13]:
prompt_word_length=train['prompt'].apply(lambda x:len(x.split()))
prompt_word_length.describe()

count    2000.00000
mean       18.14650
std         6.78189
min         3.00000
25%        14.00000
50%        17.00000
75%        22.00000
max        51.00000
Name: prompt, dtype: float64

In [14]:
prompt_char_len=train["prompt"].str.len()
prompt_char_len.describe()

count    2000.000000
mean      117.669500
std        44.408674
min        19.000000
25%        87.750000
50%       111.000000
75%       141.000000
max       337.000000
Name: prompt, dtype: float64

## Options Word Length

In [15]:
word_lens=[]
for col in ["A","B","C","D","E"]:
    word_lens.append(train[col].str.split().str.len())

In [16]:
for i in range(len(word_lens)):
    print(f"Option {chr(i+ord("A"))}")
    print(word_lens[i].describe())

Option A
count    2000.000000
mean       26.146000
std        17.249651
min         1.000000
25%        13.000000
50%        22.000000
75%        37.000000
max        81.000000
Name: A, dtype: float64
Option B
count    2000.000000
mean       26.516000
std        18.653893
min         1.000000
25%        12.000000
50%        23.000000
75%        36.000000
max       118.000000
Name: B, dtype: float64
Option C
count    2000.00000
mean       26.54950
std        17.20987
min         1.00000
25%        14.75000
50%        24.00000
75%        36.00000
max        82.00000
Name: C, dtype: float64
Option D
count    2000.000000
mean       25.999500
std        17.098909
min         1.000000
25%        15.000000
50%        22.000000
75%        36.000000
max        78.000000
Name: D, dtype: float64
Option E
count    2000.000000
mean       26.187000
std        17.852658
min         1.000000
25%        13.000000
50%        23.000000
75%        36.000000
max       105.000000
Name: E, dtype: float64


## Correct Answer Word Length

In [17]:
correct_lengths=[]
for _, row in train.iterrows():
    correct_lengths.append(len(row[row["answer"]].split()))

pd.Series(correct_lengths).describe()

count    2000.000000
mean       28.659000
std        18.331005
min         1.000000
25%        16.000000
50%        28.000000
75%        40.000000
max       105.000000
dtype: float64

## Categories of Questions

In [34]:
zs=pipeline("zero-shot-classification",model="MoritzLaurer/ModernBERT-large-zeroshot-v2.0")
zs

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/792M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

ZeroShotClassificationPipeline: {'model': 'ModernBertForSequenceClassification', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text'}

In [35]:
results = zs(
    train["prompt"].tolist(),
    candidate_labels=["Physics","Chemistry","Biology","Mathematics","Computer Science","Engineering","Medicine","Astronomy","Philosophy","History","Economics","Politics","Geography","Language and Literature","Religion","General Knowledge"],
    batch_size=16,
    multi_label=False
)

cats=[result["labels"][0] for result in results]
cats[:5]

W0711 12:24:31.210000 58 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


['Philosophy', 'Physics', 'Physics', 'Philosophy', 'Physics']

In [36]:
train["category"]=cats
train["category"].value_counts()

category
Physics                    642
Mathematics                308
Astronomy                  200
General Knowledge          178
Engineering                106
Biology                    105
Chemistry                   76
Computer Science            55
History                     44
Language and Literature     34
Economics                   23
Geography                   22
Philosophy                  11
Medicine                    10
Politics                     3
Name: count, dtype: int64

# Preprocessing

In [15]:
def clean_text(df):
    for col in test.columns:
        if col!="answer":
            df[f"clean_{col}"]=df[col].str.lower().apply(lambda x: str(x).translate(str.maketrans("", "", string.punctuation)))
            df[f"clean_{col}"]=df[f"clean_{col}"].apply(lambda x: re.sub(r"\s+", " ", x).strip())
    return df
    

In [16]:
train=clean_text(train)
train.head()

,prompt,A,B,C,D,E,answer,clean_prompt,clean_A,clean_B,clean_C,clean_D,clean_E
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
1,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is acceleratorbased lightion fusion,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...
2,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,determine the correct option what is the term ...,blueshifting,redshifting,reddening,whitening,yellowing
3,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,select the most accurate option what is martin...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
4,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,identify the correct statement what is the con...,simultaneity is relative meaning that two even...,simultaneity is relative meaning that two even...,simultaneity is absolute meaning that two even...,simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...


In [17]:
train["clean_combined_text"]=(
    "Question: "+train["clean_prompt"] +
    "\nA: "+train["clean_A"] +
    "\nB: "+train["clean_B"] +
    "\nC: "+train["clean_C"] +
    "\nD: "+train["clean_D"] +
    "\nE: "+train["clean_E"]
)

print(train["clean_combined_text"][0])

Question: pick the best possible answer what is martin heideggers view on the relationship between time and human existence among the listed options
A: martin heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end the relationship to the past involves acknowledging it as a historical era and the relationship to the future involves creating a world that will endure beyond ones own time
B: martin heidegger believes that humans do not exist inside time but that they are time the relationship to the past is a present awareness of having been and the relationship to the future involves anticipating a potential possibility task or engagement
C: martin heidegger does not believe in the existence of time or that it has any effect on human consciousness the relationship to the past and the future is insignificant and human existence is solely based on the present
D: martin heidegger believes that the relationship between time a

## Corpus for vocabulary

In [15]:
text=' '.join(train['clean_combined_text'].tolist())
words=text.split()
print("Total words in corpus:",len(words))
print("Total unique words in corpus:",len(set(words)))

Total words in corpus: 282930
Total unique words in corpus: 3102


In [16]:
word_freq=Counter(words)
print(f"Most Frequent Words:")
for i,j in (word_freq.most_common(10)):
    print(f"{i}: {j}")
print(f"\nLeast Frequent Words:")
for i,j in (word_freq.most_common()[-11:-1][::-1]):
    print(f"{i}: {j}")

Most Frequent Words:
the: 24709
of: 13223
a: 10729
is: 9327
and: 6326
in: 6131
to: 5771
that: 4037
by: 2015
an: 1843

Least Frequent Words:
subsystems: 1
increases: 1
subsystem: 2
ecosystem: 2
decreases: 2
subframework: 2
lowers: 2
submechanisms: 2
differences: 2
boosts: 3


## Split

In [18]:
#train_df,val_df=train_test_split(train,test_size=0.2,stratify=train["answer"],random_state=42)

skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

#print(train_df["answer"].value_counts()/len(train_df))
#print(val_df["answer"].value_counts()/len(val_df))

# Scoring

In [19]:
def map_at_3(true,pred):
    scores=[]
    for actual,preds in zip(true,pred):
        score=0.0
        for rank,pred in enumerate(preds,start=1):
            if pred==actual:
                score=1.0/rank
                break
        scores.append(score)
    return np.mean(scores)

In [20]:
def top3_accuracy(y_true, predictions):
    return np.mean([truth in pred for truth, pred in zip(y_true, predictions)])

In [21]:
def top1_accuracy(y_true, predictions):
    top1=[pred[0] if len(pred) else "Z" for pred in predictions]

    return accuracy_score(y_true,top1)

# PreTrained Models

## Zero-Shot Classification

In [ ]:
'''qwen_model_name="Qwen/Qwen2.5-7B-Instruct"
qwen_tokenizer=AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model=AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    dtype=torch.float16,
    device_map="auto"
)'''

In [ ]:
'''def create_zero_shot_prompt(row):
    zero_shot_prompt=f"""
    You are solving a multiple choice question containing 5 choices.
    Question:
    {row["prompt"]}
    Choices:
    A. {row["A"]}
    B. {row["B"]}
    C. {row["C"]}
    D. {row["D"]}
    E. {row["E"]}
    Return ONLY the three most likely answer labels with a single space separating them.
    Example:
    C A D
    """
    return zero_shot_prompt'''

In [ ]:
'''def create_few_shot_prompt(row):
    few_shot_examples = []
    for label in ["A", "B", "C", "D", "E"]:
        example = train_df[train_df["answer"] == label].sample(n=1,random_state=42).iloc[0]
        few_shot_examples.append(example)
    prompt = """You are an expert at solving multiple-choice questions.
                Below are some solved examples.\n"""

    for i, ex in enumerate(few_shot_examples, 1):

        prompt += f"""Example {i}
        Question: {ex["prompt"]}
        
        Choices:
        A. {ex["A"]}
        B. {ex["B"]}
        C. {ex["C"]}
        D. {ex["D"]}
        E. {ex["E"]}
        
        Correct Answer:
        {ex["answer"]}
        
        """
        
    prompt += f"""
    Now answer the following question.
    
    Question:
    {row["prompt"]}
    
    Choices:
    A. {row["A"]}
    B. {row["B"]}
    C. {row["C"]}
    D. {row["D"]}
    E. {row["E"]}
    
    Rank the choices based on their probability of being the correct answer.
    Then return the top three answer labels separated by spaces.
    
    Example output:
    C A D
    
    Do not explain your answer.
    """

    return prompt'''

In [ ]:
'''def predict_qwen(row):

    prompt = create_few_shot_prompt(row)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    ).to(qwen_model.device)

    with torch.no_grad():

        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id
        )

    generated = outputs[0][inputs.input_ids.shape[1]:]

    prediction = qwen_tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    labels = re.findall(r"\b[A-E]\b", prediction)

    return labels[:3]'''

In [ ]:
'''predictions = []

fold_map = []
fold_acc = []
fold_top3 = []

for fold,(train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):
    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)
    predictions=[]
    for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
        pred = predict_qwen(row)
        predictions.append(pred)
        torch.cuda.empty_cache()
    map3 = map_at_3(
        val_df["answer"],
        predictions
    )
    
    acc = top1_accuracy(
        val_df["answer"],
        predictions
    )
    
    top3 = top3_accuracy(
        val_df["answer"],
        predictions
    )
    wandb.log({
        "Fold": fold + 1,
        "MAP@3": map3,
        "Accuracy": acc,
        "Top3 Accuracy": top3
    })
    fold_map.append(map3)
    fold_acc.append(acc)
    fold_top3.append(top3)
print(f"MAP@3 : {np.mean(fold_map):.4f}")
print(f"Accuracy : {np.mean(fold_acc):.4f}")
print(f"Top3 Accuracy : {np.mean(fold_top3):.4f}")'''

In [ ]:
'''wandb.log({
    "Average MAP@3": np.mean(fold_map),
    "Average Accuracy": np.mean(fold_acc),
    "Average Top3 Accuracy": np.mean(fold_top3)
})

wandb.finish()'''

# Model from Scratch

## BiLSTM + Attention Scores

# Model of Choice

## Tf-Idf + Logistic Regression

In [21]:
'''def build_pairwise_dataset(df,test=False):
    rows=[]
    options=["A","B","C","D","E"]
    for qid,row in df.iterrows():
        for c in options:
            if not test:
                rows.append({
                    "text":"Question: "+row["clean_prompt"]+"\nAnswer: "+row[f"clean_{c}"],
                    "label":1 if c==row["answer"] else 0,
                    "question_id":qid,
                    "option":c
                })
            else:
                rows.append({
                    "text":"Question: "+row["clean_prompt"]+"\nAnswer: "+row[f"clean_{c}"],
                    "question_id":qid,
                    "option":c
                })

    return pd.DataFrame(rows)'''

In [22]:
'''vectorizer=TfidfVectorizer(
    ngram_range=(1,2),
    stop_words="english",
    min_df=3,
    sublinear_tf=True
)
clf=LogisticRegression(max_iter=1000,class_weight="balanced",random_state=42)'''

In [24]:
'''fold_map3=[]
fold_acc=[]
fold_top3=[]

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):
    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)
    
    print("Fold:",fold+1)
    
    pair_train=build_pairwise_dataset(train_df)
    pair_val=build_pairwise_dataset(val_df)

    X_train=vectorizer.fit_transform(pair_train["text"])
    X_val=vectorizer.transform(pair_val["text"])

    clf.fit(X_train,pair_train["label"])
    
    probs=clf.predict_proba(X_val)[:,1]

    predictions=[]
    choices=["A","B","C","D","E"]
    for i in range(len(val_df)):
        start=i*5
        end=start+5
        scores=probs[start:end]
        ranked=np.argsort(scores)[::-1]
        top3=[choices[j] for j in ranked[:3]]
        predictions.append(top3)
       
    map3=map_at_3(val_df["answer"],predictions)
    acc=top1_accuracy(val_df["answer"],predictions)
    top3=top3_accuracy(val_df["answer"],predictions)

    print(f"MAP@3: {map3:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Top3 Accuracy: {top3:.4f}")

    fold_map3.append(map3)
    fold_acc.append(acc)
    fold_top3.append(top3)

    wandb.log({
        "Fold": fold + 1,
        "MAP@3": map3,
        "Top1 Accuracy": acc,
        "Top3 Accuracy": top3
    })'''

Fold: 1
MAP@3: 0.9721
Accuracy: 0.9533
Top3 Accuracy: 0.9945
Fold: 2
MAP@3: 0.9634
Accuracy: 0.9368
Top3 Accuracy: 0.9945
Fold: 3
MAP@3: 0.9646
Accuracy: 0.9394
Top3 Accuracy: 0.9972
Fold: 4
MAP@3: 0.9720
Accuracy: 0.9532
Top3 Accuracy: 0.9945
Fold: 5
MAP@3: 0.9596
Accuracy: 0.9311
Top3 Accuracy: 0.9917


'    wandb.log({\n        "Fold": fold + 1,\n        "MAP@3": map3,\n        "Top1 Accuracy": acc,\n        "Top3 Accuracy": top3\n    })'

In [25]:
'''print("Average MAP@3:", np.mean(fold_map3))
print("Average Accuracy:", np.mean(fold_acc))
print("Average Top3 Accuracy:", np.mean(fold_top3))

wandb.log({
    "Average MAP@3": np.mean(fold_map3),
    "Average Top1 Accuracy": np.mean(fold_acc),
    "Average Top3 Accuracy": np.mean(fold_top3)
})

wandb.finish()'''

Average MAP@3: 0.966334927698564
Average Accuracy: 0.9427602700329973
Average Top3 Accuracy: 0.994496412678231


'wandb.log({\n    "Average MAP@3": np.mean(fold_map3),\n    "Average Top1 Accuracy": np.mean(fold_acc),\n    "Average Top3 Accuracy": np.mean(fold_top3)\n})\n\nwandb.finish()'

In [26]:
'''pair_train=build_pairwise_dataset(train)
X_train=vectorizer.fit_transform(pair_train["text"])
clf.fit(X_train,pair_train["label"])'''

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [27]:
'''test=clean_text(test)
test.head()'''

,prompt,A,B,C,D,E,clean_prompt,clean_A,clean_B,clean_C,clean_D,clean_E
0,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",pick the best possible answer what is the rela...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...
1,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi...",what is the estimated redshift of ceers93316 a...,approximately z 60 corresponding to 1 billion ...,approximately z 167 corresponding to 2358 mill...,approximately z 30 corresponding to 5 billion ...,approximately z 100 corresponding to 13 billio...,approximately z 130 corresponding to 30 billio...
2,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...,pick the best possible answer what is the reas...,the sun appears yellowish due to a reflection ...,the longer wavelengths of light such as red an...,the sun appears yellowish due to the scatterin...,the sun emits a yellow light due to its own sp...,the atmosphere absorbs the shorter wavelengths...
3,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,what is the significance of the redshiftdistan...,observations of the redshiftdistance relations...,observations of the redshiftdistance relations...,observations of the redshiftdistance relations...,observations of the redshiftdistance relations...,observations of the redshiftdistance relations...
4,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,what is the landaulifshitzgilbert equation use...,the landaulifshitzgilbert equation is a differ...,the landaulifshitzgilbert equation is a differ...,the landaulifshitzgilbert equation is a differ...,the landaulifshitzgilbert equation is a differ...,the landaulifshitzgilbert equation is a differ...


In [28]:
'''pair_test=build_pairwise_dataset(test,True)
X_test=vectorizer.transform(pair_test["text"])
probs=clf.predict_proba(X_test)[:, 1]
probs'''

array([0.59217922, 0.36603222, 0.37571282, ..., 0.80480562, 0.22994401,
       0.22156312], shape=(2500,))

In [29]:
'''choices=["A","B","C","D","E"]
test_predictions=[]
for i in range(len(test)):
    scores=probs[i*5:(i+1)*5]
    ranked=np.argsort(scores)[::-1]
    top3=[choices[j] for j in ranked[:3]]
    test_predictions.append(" ".join(top3))
test_predictions[:5]'''

['A E C', 'B C E', 'B E D', 'E C A', 'C A D']

## RAG System

In [22]:
wiki=load_dataset("wikimedia/wikipedia","20231101.en",split="train")
wiki=wiki.select(range(50000))

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

20231101.en/train-00000-of-00041.parquet:   0%|          | 0.00/420M [00:00<?, ?B/s]

20231101.en/train-00001-of-00041.parquet:   0%|          | 0.00/351M [00:00<?, ?B/s]

20231101.en/train-00002-of-00041.parquet:   0%|          | 0.00/329M [00:00<?, ?B/s]

...

20231101.en/train-00038-of-00041.parquet:   0%|          | 0.00/538M [00:00<?, ?B/s]

20231101.en/train-00039-of-00041.parquet:   0%|          | 0.00/465M [00:00<?, ?B/s]

20231101.en/train-00040-of-00041.parquet:   0%|          | 0.00/422M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6407814 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

In [23]:
def clean_data(text):
    text=text.lower()
    text=re.sub(r"\s+"," ",text)
    text=re.sub(r"\[[^\]]*\]","",text)
    text=text.strip()
    return text

In [24]:
chunker=TokenChunker(chunk_size=CONFIG["chunk_size"],chunk_overlap=CONFIG["chunk_overlap"])
chunker

TokenChunker(tokenizer=<chonkie.tokenizer.ChonkieAutoTokenizer object at 0x7ad401771c10>, chunk_size=256, chunk_overlap=32)

In [25]:
model = SentenceTransformer("BAAI/bge-base-en-v1.5") 
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [28]:
kb=[]
chunk_records=[]
for article in wiki:
    title=article["title"]
    text=clean_data(article["text"])
    url=article.get("url", "")

    if len(text.strip()) == 0:
        continue

    chunks=chunker(text)

    for chunk_idx, chunk in enumerate(chunks):
        chunk_text=chunk.text
        kb.append(chunk_text)
        chunk_records.append({
            "chunk_id":str(uuid.uuid4()),
            "chunk_index": chunk_idx,
            "title": title,
            "url": url,
            "text": chunk_text,
            "token_count": chunk.token_count
        })

print(f"Total chunks generated: {len(chunk_records)}")

Total chunks generated: 1180030


In [34]:
with jsonlines.open('/kaggle/working/knowledge_base.jsonl', mode='w') as writer:
    writer.write_all(chunk_records)

wandb.config.update({"total_chunks": len(chunk_records)})

print(f"Chunk and metadata saved")

Chunk and metadata saved


In [ ]:
texts_to_embed=[chunk["text"] for chunk in chunk_records]
embeddings=model.encode(texts_to_embed)

In [ ]:
index_path="/kaggle/working/vector_index.idx"

embedding_dimension=embeddings.shape[1]
faiss_index=faiss.IndexFlatL2(embedding_dimension) 
faiss_index.add(embeddings)

faiss.write_index(faiss_index, index_path)

wandb.config.update({"embedding_dimension": embedding_dimension})

print(f"Saved FAISS index to {index_path}")

In [ ]:
print("Logging data and index to Weights & Biases...")

artifact = wandb.Artifact(
    name="wikipedia_knowledge_base",
    type="dataset",
    description="JSONL chunks and FAISS index for Wikipedia Dataset",
    metadata={
        "chunking_strategy": "TokenChunker",
        "chunk_size_tokens": CONFIG["chunk_size"],
        "chunk_overlap_tokens": CONFIG["chunk_overlap"],
        "total_chunks_generated": len(chunk_records),
        
        "embedding_model": CONFIG["embedding_model"],
        "embedding_dimension": embedding_dimension,
        "vector_index_type": "FAISS IndexFlatL2",
        
        "source_material_type": "Wikipedia",
    }
)

artifact.add_file("/kaggle/working/knowledge_base.jsonl")
artifact.add_file("/kaggle/working/vector_index.idx")
wandb.log_artifact(artifact)
wandb.finish()

print("Chunking and Indexing complete and pushed to W&B")

In [ ]:
'''def rag_predict(row,kb,index):
    prompt_row=row['prompt']
    
    row_embeddings=np.expand_dims(model.encode(prompt_row, show_progress_bar=False),axis=0)
    row_docs=index.search(row_embeddings, k=5)[1][0]

    docs_5=[kb[i] for i in row_docs] 
    pairs=[[prompt_row, doc] for doc in docs_5] 
    ce_scores=cross_encoder.predict(pairs) 

    best_doc=kb[np.argsort(-ce_scores)[0]]
    aug_string=f"Context: {best_doc} Question: {prompt_row}"
    choices=["A","B","C","D","E"]
    row_labels=[row[i] for i in choices]
    
    zs_pred=zs(aug_string,candidate_labels=row_labels)
    zs_labels=zs_pred['labels'][:3]
    
    top3_preds=[]
    for i in zs_labels:
        top3_preds.append(choices[row_labels.index(i)])

    return top3_preds'''

In [ ]:
'''fold_map3=[]
fold_acc=[]
fold_top3=[]

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):
    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)
    
    print("Fold:",fold+1)
    
    kb,index=build_kb(train_df)

    predictions=[]
    for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
        top3_preds=rag_predict(row,kb,index)        
        predictions.append(top3_preds)
       
    map3=map_at_3(val_df["answer"],predictions)
    acc=top1_accuracy(val_df["answer"],predictions)
    top3=top3_accuracy(val_df["answer"],predictions)

    print(f"MAP@3: {map3:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Top3 Accuracy: {top3:.4f}")

    fold_map3.append(map3)
    fold_acc.append(acc)
    fold_top3.append(top3)

    wandb.log({
        "Fold": fold + 1,
        "MAP@3": map3,
        "Top1 Accuracy": acc,
        "Top3 Accuracy": top3
    })
'''

In [ ]:
'''print("Average MAP@3:", np.mean(fold_map3))
print("Average Accuracy:", np.mean(fold_acc))
print("Average Top3 Accuracy:", np.mean(fold_top3))

wandb.log({
    "Average MAP@3": np.mean(fold_map3),
    "Average Top1 Accuracy": np.mean(fold_acc),
    "Average Top3 Accuracy": np.mean(fold_top3)
})

wandb.finish()'''

In [ ]:
'''test_predictions=[]
kb_train,index_train=build_kb(train)
for _, row in tqdm(test.iterrows(), total=len(test)):
    test_predictions.append(rag_predict(row,kb_train,index_train))
test_predictions[:5]'''

# Submission

In [30]:
submission_preds = [" ".join(pred) for pred in test_predictions]
sub=pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
sub.set_index('ID',inplace=True)
sub.head()

,Prediction
ID,
1,A B C
2,A B C
3,A B C
4,A B C
5,A B C


In [31]:
sub['Prediction']=submission_preds
sub.head()

,Prediction
ID,
1,A E C
2,B C E
3,B E D
4,E C A
5,C A D


In [32]:
sub.to_csv("submission.csv")
print("Submission File Created")

Submission File Created
